# 09 — Best-Model Fine-Tuning

**Goals:**
1. Load every checkpoint produced by notebooks 02–08, evaluate on the validation set, and automatically select the best-performing model.
2. Fine-tune it with advanced techniques:
   - **Asymmetric Loss (ASL)** — suppresses easy-negative noise in multi-label tasks.
   - **Weighted random sampling** — up-samples rare-label combinations during training.
   - **Two-stage fine-tuning** (pretrained models only): Stage A freezes the backbone and trains the head; Stage B unfreezes the full network with a lower backbone LR.
3. **Per-class threshold tuning** on the validation set to maximise per-class F1.
4. **Test-time augmentation (TTA)** — average predictions over multiple random crops/flips.
5. Final test-set evaluation and **GradCAM saliency visualisations**.

In [ ]:
import sys, time, copy
from pathlib import Path

# Make shared utilities importable
sys.path.insert(0, str(Path("../experiments").resolve()))
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import models as tv_models, transforms

from utils import (
    LABEL_ORDER, METRIC_KEYS, NORM_MEAN, NORM_STD,
    set_seed, load_dataset, split_dataset,
    TransformSubset, get_train_transform, get_eval_transform, build_dataloaders,
    compute_multilabel_metrics, print_metric_table,
    save_checkpoint, print_model_info,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, plot_per_class_examples,
    plot_training_history, denorm, labels_to_text,
)

# ── Global config ─────────────────────────────────────────────────────────────
SEED         = 42
SPLIT        = [0.7, 0.15, 0.15]
BASE_DIR     = "../data/aggregated"
CKPT_DIR     = Path("../checkpoints")
NUM_LABELS   = len(LABEL_ORDER)
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

set_seed(SEED)
print(f"Device: {DEVICE}  |  Labels: {NUM_LABELS}")
print(f"Label order: {LABEL_ORDER}")

## 1 · Model Registry & Checkpoint Comparison

We define every architecture from notebooks 02–08 so we can load the corresponding `final_*.pth` checkpoints and compare them on the validation set.

In [ ]:
# ── Re-define all architectures (must exactly match training notebooks) ───────

# ── 02: SmallCNN (scratch, 128×128) ──────────────────────────────────────────
class SmallCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding="same"), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding="same"), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding="same"), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding="same"), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(512, 128), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# ── 03: VGGScratch (scratch, 128×128) ─────────────────────────────────────────
def _vgg_block(in_ch, out_ch, n_convs):
    layers = []
    for i in range(n_convs):
        layers += [nn.Conv2d(in_ch if i == 0 else out_ch, out_ch, 3, padding=1),
                   nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)]
    layers.append(nn.MaxPool2d(2, 2))
    return nn.Sequential(*layers)

class VGGScratch(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            _vgg_block(3,   64,  2),
            _vgg_block(64,  128, 2),
            _vgg_block(128, 256, 3),
            _vgg_block(256, 512, 3),
            _vgg_block(512, 512, 3),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 2048), nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(2048, 512),         nn.ReLU(inplace=True), nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# ── 08: ViT (scratch, 128×128) ────────────────────────────────────────────────
class VisionTransformer(nn.Module):
    def __init__(self, num_classes, image_size=128, patch_size=16,
                 embed_dim=256, num_heads=8, depth=6, mlp_dim=512, dropout=0.1):
        super().__init__()
        assert image_size % patch_size == 0
        self.patch_size  = patch_size
        self.num_patches = (image_size // patch_size) ** 2
        patch_dim        = 3 * patch_size * patch_size
        self.patch_embed = nn.Linear(patch_dim, embed_dim)
        self.cls_token   = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_embed   = nn.Parameter(torch.randn(1, self.num_patches + 1, embed_dim))
        self.dropout     = nn.Dropout(dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=mlp_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(embed_dim, num_classes))
    def forward(self, x):
        B, C, H, W = x.shape
        ps      = self.patch_size
        patches = x.unfold(2, ps, ps).unfold(3, ps, ps)
        patches = patches.contiguous().view(B, C, -1, ps, ps)
        patches = patches.permute(0, 2, 1, 3, 4).flatten(2)
        x       = self.patch_embed(patches)
        cls     = self.cls_token.expand(B, -1, -1)
        x       = torch.cat((cls, x), dim=1)
        x       = self.dropout(x + self.pos_embed)
        x       = self.transformer(x)
        return self.head(self.norm(x[:, 0]))

# ── Pretrained model factory functions (04–07) ────────────────────────────────
def _make_vgg16(n):     m = tv_models.vgg16_bn(weights=None);         m.classifier[-1] = nn.Linear(4096, n);              return m
def _make_r50(n):       m = tv_models.resnet50(weights=None);          m.fc = nn.Linear(m.fc.in_features, n);              return m
def _make_mnv2(n):      m = tv_models.mobilenet_v2(weights=None);      m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n); return m
def _make_efb0(n):      m = tv_models.efficientnet_b0(weights=None);   m.classifier[-1] = nn.Linear(m.classifier[-1].in_features, n); return m

# ── Registry ──────────────────────────────────────────────────────────────────
# Each entry: create_fn, image_size, is_pretrained_backbone,
#             backbone_attr (for freeze/unfreeze), head_attr, gradcam_layer_fn
MODEL_REGISTRY = {
    "small_cnn": {
        "create_fn":    lambda n: SmallCNN(n),
        "image_size":   128,
        "pretrained":   False,
        "backbone_attr": "features",
        "head_attr":    "classifier",
        "gradcam_fn":   lambda m: m.features[12],  # last Conv2d
    },
    "vgg_scratch": {
        "create_fn":    lambda n: VGGScratch(n),
        "image_size":   128,
        "pretrained":   False,
        "backbone_attr": "features",
        "head_attr":    "classifier",
        "gradcam_fn":   lambda m: m.features[-1][-3],  # last Conv2d in last vgg_block
    },
    "vgg16_pretrained": {
        "create_fn":    _make_vgg16,
        "image_size":   224,
        "pretrained":   True,
        "backbone_attr": "features",
        "head_attr":    "classifier",
        "gradcam_fn":   lambda m: m.features[-3],  # last Conv2d before MaxPool
    },
    "resnet50_pretrained": {
        "create_fn":    _make_r50,
        "image_size":   224,
        "pretrained":   True,
        "backbone_attr": None,  # handled separately via named children
        "head_attr":    "fc",
        "gradcam_fn":   lambda m: m.layer4[-1],
    },
    "mobilenetv2_pretrained": {
        "create_fn":    _make_mnv2,
        "image_size":   224,
        "pretrained":   True,
        "backbone_attr": "features",
        "head_attr":    "classifier",
        "gradcam_fn":   lambda m: m.features[-1],
    },
    "efficientnet_b0_pretrained": {
        "create_fn":    _make_efb0,
        "image_size":   224,
        "pretrained":   True,
        "backbone_attr": "features",
        "head_attr":    "classifier",
        "gradcam_fn":   lambda m: m.features[-1],
    },
    "vit_scratch": {
        "create_fn":    lambda n: VisionTransformer(num_classes=n, image_size=128),
        "image_size":   128,
        "pretrained":   False,
        "backbone_attr": "transformer",
        "head_attr":    "head",
        "gradcam_fn":   None,   # ViT: use gradient saliency instead
    },
}

print(f"Registry contains {len(MODEL_REGISTRY)} architectures:")
for name, cfg in MODEL_REGISTRY.items():
    ckpt = CKPT_DIR / f"final_{name}.pth"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"  {name:<28}  img={cfg['image_size']}  {status}")

In [ ]:
# ── Load each available checkpoint and evaluate on val (224-based loader for
#    pretrained, 128-based loader for scratch).
# ── We keep separate val loaders per image_size to avoid redundant resampling.

full_dataset = load_dataset(BASE_DIR)
train_raw, val_raw, test_raw = split_dataset(full_dataset, SPLIT, SEED)
print(f"Train: {len(train_raw)}  |  Val: {len(val_raw)}  |  Test: {len(test_raw)}")

_val_loaders = {}
for img_sz in (128, 224):
    _val_loaders[img_sz] = DataLoader(
        TransformSubset(val_raw, get_eval_transform(img_sz)),
        batch_size=64, shuffle=False, num_workers=2, pin_memory=True,
    )

def _eval_checkpoint(model, val_loader, device):
    model.eval().to(device)
    ll, lp, lpr, llg = [], [], [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            logits = model(imgs.to(device))
            probs  = torch.sigmoid(logits)
            preds  = (probs >= 0.5).float()
            ll.append(labels); lp.append(preds.cpu())
            lpr.append(probs.cpu()); llg.append(logits.cpu())
    return compute_multilabel_metrics(
        torch.cat(ll), torch.cat(lp),
        probs=torch.cat(lpr), logits=torch.cat(llg),
    )

comparison_rows = []
for arch, cfg in MODEL_REGISTRY.items():
    ckpt_path = CKPT_DIR / f"final_{arch}.pth"
    if not ckpt_path.exists():
        print(f"  [SKIP] {arch}: checkpoint not found")
        continue
    model = cfg["create_fn"](NUM_LABELS)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
    metrics = _eval_checkpoint(model, _val_loaders[cfg["image_size"]], DEVICE)
    comparison_rows.append({"arch": arch, **{k: round(metrics[k], 4) for k in METRIC_KEYS}})
    print(f"  {arch:<28}  val_F1={metrics['f1_micro']:.4f}  exact={metrics['exact_match']:.4f}")

df_cmp = pd.DataFrame(comparison_rows).set_index("arch")
df_cmp = df_cmp.sort_values("f1_micro", ascending=False)
print("\n=== Checkpoint comparison (val set) ===")
print(df_cmp[["f1_micro", "exact_match", "hamming_acc", "mean_iou",
              "precision_micro", "recall_micro", "loss"]].to_string())

BEST_ARCH = df_cmp.index[0]
print(f"\n>>> Best model: {BEST_ARCH}  (val F1={df_cmp.loc[BEST_ARCH, 'f1_micro']:.4f})")

## 2 · Advanced Fine-Tuning Setup

We prepare the building blocks for fine-tuning:
- **AsymmetricLoss** — down-weights easy negatives via asymmetric focusing.
- **Weighted sampler** — assigns each training sample a weight inversely proportional to the mean positive-label frequency among its active labels, so rare combinations are sampled more often.
- **Backbone / head parameter helpers** — used to set up per-group learning rates.

In [ ]:
# ── Asymmetric Loss ───────────────────────────────────────────────────────────
class AsymmetricLoss(nn.Module):
    """ASL (Ridnik et al. 2021): asymmetric focusing for multi-label classification."""

    def __init__(self, gamma_neg: float = 4.0, gamma_pos: float = 1.0,
                 clip: float = 0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs     = torch.sigmoid(logits)
        probs_neg = (probs - self.clip).clamp(min=0.0)  # probability margin shift

        loss_pos = -targets       * (1 - probs)    ** self.gamma_pos * torch.log(probs.clamp(1e-8))
        loss_neg = -(1 - targets) * probs_neg      ** self.gamma_neg * torch.log((1 - probs_neg).clamp(1e-8))
        return (loss_pos + loss_neg).mean()


# ── Weighted sampler ──────────────────────────────────────────────────────────
def make_weighted_sampler(train_raw) -> WeightedRandomSampler:
    """Per-sample weight = 1 / mean positive-label frequency for its active labels."""
    print("Computing label frequencies for weighted sampler...")
    all_labels = torch.stack([train_raw[i][1] for i in range(len(train_raw))])
    pos_freq   = all_labels.float().mean(dim=0).clamp(min=1e-6)  # (NUM_LABELS,)

    weights = []
    for i in range(len(all_labels)):
        active = all_labels[i].bool()
        w = (1.0 / pos_freq[active].mean().item()) if active.any() else 1.0
        weights.append(w)
    weights = torch.tensor(weights, dtype=torch.float)
    print(f"  weight range: {weights.min():.2f} – {weights.max():.2f}")
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


# ── Backbone / head parameter splitting ──────────────────────────────────────
def get_param_groups(model: nn.Module, arch: str):
    """Return (backbone_params, head_params) for two-stage fine-tuning."""
    cfg       = MODEL_REGISTRY[arch]
    head_attr = cfg["head_attr"]
    head_mod  = getattr(model, head_attr)
    head_ids  = {id(p) for p in head_mod.parameters()}
    head_params     = list(head_mod.parameters())
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    return backbone_params, head_params


def freeze_backbone(model: nn.Module, arch: str) -> None:
    backbone_params, _ = get_param_groups(model, arch)
    for p in backbone_params:
        p.requires_grad = False


def unfreeze_all(model: nn.Module) -> None:
    for p in model.parameters():
        p.requires_grad = True


print("AsymmetricLoss, weighted sampler, and parameter helpers defined.")

## 3 · Two-Stage Fine-Tuning

| Stage | Trainable params | LR | Epochs | Purpose |
|-------|-----------------|-----|--------|---------|
| A | head only | 1e-3 | up to 10 | Adapt classifier to ASL + weighted data |
| B | full model | backbone 1e-5, head 1e-4 | up to 30 | End-to-end refinement |

For from-scratch models (no ImageNet backbone) Stage A is skipped and we go straight to Stage B with a uniform LR.

In [ ]:
# ── Shared fine-tuning training loop ──────────────────────────────────────────
def _run_finetune(
    model: nn.Module,
    train_loader,
    val_loader,
    optimizer,
    criterion: nn.Module,
    device,
    max_epochs: int,
    warmup_epochs: int = 2,
    early_stop_patience: int = 5,
    lr_reduce_patience: int = 3,
    grad_clip: float = 1.0,
    threshold: float = 0.5,
    stage_label: str = "",
):
    """Train model in-place; return (best_state_dict, best_val_f1, history, epochs_run)."""
    base_lrs = [pg["lr"] for pg in optimizer.param_groups]
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=lr_reduce_patience
    )
    history     = {k: {"train": [], "val": []} for k in METRIC_KEYS}
    best_val_f1 = -1.0
    best_state  = None
    no_improve  = 0

    for epoch in range(1, max_epochs + 1):
        # Warmup
        if epoch <= warmup_epochs:
            scale = epoch / warmup_epochs
            for pg, base_lr in zip(optimizer.param_groups, base_lrs):
                pg["lr"] = base_lr * scale

        # ── Train ──────────────────────────────────────────────────────────────
        model.train()
        tr_l, tr_p, tr_pr, tr_lg = [], [], [], []
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            logits = model(images)
            loss   = criterion(logits, targets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                preds = (probs >= threshold).float()
            tr_l.append(targets.cpu());  tr_p.append(preds.cpu())
            tr_pr.append(probs.cpu());   tr_lg.append(logits.detach().cpu())
        train_m = compute_multilabel_metrics(
            torch.cat(tr_l), torch.cat(tr_p),
            probs=torch.cat(tr_pr), logits=torch.cat(tr_lg),
        )

        # ── Validate ───────────────────────────────────────────────────────────
        model.eval()
        val_l, val_p, val_pr, val_lg = [], [], [], []
        with torch.no_grad():
            for images, labels in val_loader:
                logits = model(images.to(device))
                probs  = torch.sigmoid(logits)
                preds  = (probs >= threshold).float()
                val_l.append(labels.cpu());   val_p.append(preds.cpu())
                val_pr.append(probs.cpu());   val_lg.append(logits.cpu())
        val_m = compute_multilabel_metrics(
            torch.cat(val_l), torch.cat(val_p),
            probs=torch.cat(val_pr), logits=torch.cat(val_lg),
        )

        for k in METRIC_KEYS:
            history[k]["train"].append(train_m[k])
            history[k]["val"].append(val_m[k])

        val_f1 = val_m["f1_micro"]
        if epoch > warmup_epochs:
            scheduler.step(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state  = copy.deepcopy(model.state_dict())
            no_improve  = 0
        else:
            no_improve += 1

        lr_str = "/".join(f"{pg['lr']:.1e}" for pg in optimizer.param_groups)
        print(f"[{stage_label}] Epoch {epoch:>2}/{max_epochs}  lr={lr_str}  "
              f"train_F1={train_m['f1_micro']:.4f}  val_F1={val_f1:.4f}"
              + ("  *" if no_improve == 0 else ""))

        if no_improve >= early_stop_patience:
            print(f"  [Early stop] No val F1 improvement for {early_stop_patience} epochs.")
            break

    model.load_state_dict(best_state)
    return best_state, best_val_f1, history, epoch

print("Fine-tuning loop defined.")

In [ ]:
# ── Load best checkpoint and build data loaders ───────────────────────────────
cfg        = MODEL_REGISTRY[BEST_ARCH]
IMAGE_SIZE = cfg["image_size"]
IS_PRETRAINED = cfg["pretrained"]
BATCH_SIZE = 32 if IMAGE_SIZE == 224 else 64

print(f"Fine-tuning  arch={BEST_ARCH}  img_size={IMAGE_SIZE}  pretrained={IS_PRETRAINED}")

ckpt_path  = CKPT_DIR / f"final_{BEST_ARCH}.pth"
best_model = cfg["create_fn"](NUM_LABELS)
best_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=True))
best_model.to(DEVICE)
print_model_info(best_model)

# Build sampler and loaders
sampler       = make_weighted_sampler(train_raw)
train_tf      = get_train_transform(IMAGE_SIZE)
eval_tf       = get_eval_transform(IMAGE_SIZE)
train_ds_ft   = TransformSubset(train_raw, train_tf)
val_loader_ft = DataLoader(TransformSubset(val_raw, eval_tf),
                           batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
# Use weighted sampler — note: can't use shuffle=True with a sampler
train_loader_ft = DataLoader(train_ds_ft, batch_size=BATCH_SIZE,
                             sampler=sampler, num_workers=2, pin_memory=True)

asl = AsymmetricLoss(gamma_neg=4.0, gamma_pos=1.0, clip=0.05)
print("Data loaders and ASL criterion ready.")

In [ ]:
# ── Stage A: Head-only (pretrained models only) ───────────────────────────────
t0 = time.time()

if IS_PRETRAINED:
    print("=== Stage A: Head-only fine-tuning ===")
    freeze_backbone(best_model, BEST_ARCH)
    _, head_params = get_param_groups(best_model, BEST_ARCH)
    trainable_a = sum(p.numel() for p in best_model.parameters() if p.requires_grad)
    print(f"  Trainable params: {trainable_a:,}")

    opt_a = optim.AdamW(head_params, lr=1e-3, weight_decay=1e-4)
    state_a, f1_a, hist_a, ep_a = _run_finetune(
        best_model, train_loader_ft, val_loader_ft, opt_a, asl, DEVICE,
        max_epochs=10, warmup_epochs=2, early_stop_patience=4,
        stage_label="Stage-A",
    )
    print(f"\nStage A done — best val F1: {f1_a:.4f}  ({ep_a} epochs, {time.time()-t0:.0f}s)")
    plot_training_history(hist_a, ep_a, experiment_name=f"{BEST_ARCH} Stage-A", lr=1e-3, weight_decay=1e-4)
else:
    print(f"Skipping Stage A (scratch model: {BEST_ARCH})")
    state_a, f1_a = None, None

In [ ]:
# ── Stage B: Full model ────────────────────────────────────────────────────────
print("=== Stage B: Full-model fine-tuning ===")
unfreeze_all(best_model)
trainable_b = sum(p.numel() for p in best_model.parameters() if p.requires_grad)
print(f"  Trainable params: {trainable_b:,}")

if IS_PRETRAINED:
    backbone_params, head_params = get_param_groups(best_model, BEST_ARCH)
    opt_b = optim.AdamW([
        {"params": backbone_params, "lr": 1e-5},
        {"params": head_params,     "lr": 1e-4},
    ], weight_decay=1e-4)
else:
    opt_b = optim.AdamW(best_model.parameters(), lr=3e-4, weight_decay=1e-4)

t0 = time.time()
state_b, f1_b, hist_b, ep_b = _run_finetune(
    best_model, train_loader_ft, val_loader_ft, opt_b, asl, DEVICE,
    max_epochs=30, warmup_epochs=3, early_stop_patience=7,
    stage_label="Stage-B",
)
print(f"\nStage B done — best val F1: {f1_b:.4f}  ({ep_b} epochs, {time.time()-t0:.0f}s)")
plot_training_history(hist_b, ep_b,
                      experiment_name=f"{BEST_ARCH} Stage-B",
                      lr=1e-5 if IS_PRETRAINED else 3e-4, weight_decay=1e-4)

## 4 · Per-Class Threshold Tuning

For each label we grid-search a scalar threshold $t_c \in [0.1, 0.85]$ that maximises the per-class F1 on the **validation** set.  
The global default of 0.5 is then compared against the tuned vector.

In [ ]:
# ── Collect val probabilities ─────────────────────────────────────────────────
best_model.eval()
val_all_labels, val_all_probs = [], []
with torch.no_grad():
    for imgs, labels in val_loader_ft:
        probs = torch.sigmoid(best_model(imgs.to(DEVICE)))
        val_all_labels.append(labels)
        val_all_probs.append(probs.cpu())
val_labels = torch.cat(val_all_labels)   # (N, C)
val_probs  = torch.cat(val_all_probs)    # (N, C)

# ── Grid search per class ─────────────────────────────────────────────────────
THRESHOLD_GRID = torch.arange(0.10, 0.90, 0.05).tolist()
best_thresholds = torch.full((NUM_LABELS,), 0.5)

print(f"{'Label':<14} {'t_default':>10} {'t_tuned':>8} {'F1@0.5':>8} {'F1@tuned':>10}")
print("-" * 56)
for c, label in enumerate(LABEL_ORDER):
    best_f1, best_t = 0.0, 0.5
    f1_at_default   = 0.0
    for t in THRESHOLD_GRID:
        preds_c = (val_probs[:, c] >= t).float()
        tp = ((preds_c == 1) & (val_labels[:, c] == 1)).sum().float()
        fp = ((preds_c == 1) & (val_labels[:, c] == 0)).sum().float()
        fn = ((preds_c == 0) & (val_labels[:, c] == 1)).sum().float()
        f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).item()
        if abs(t - 0.5) < 0.01:
            f1_at_default = f1
        if f1 > best_f1:
            best_f1, best_t = f1, t
    best_thresholds[c] = best_t
    print(f"{label:<14} {'0.50':>10} {best_t:>8.2f} {f1_at_default:>8.4f} {best_f1:>10.4f}")

print(f"\nTuned thresholds: {best_thresholds.tolist()}")

## 5 · Test-Time Augmentation (TTA)

For each test image we draw **8 random augmented views** (same pipeline as training) plus 1 deterministic centre-crop, and average their sigmoid probabilities before applying the tuned thresholds.

In [ ]:
TTA_N_AUG = 8

tta_tf  = get_train_transform(IMAGE_SIZE)  # random augmentation
eval_tf = get_eval_transform(IMAGE_SIZE)    # deterministic

def evaluate_tta(model, raw_subset, n_aug, threshold_vec, device):
    """TTA evaluation over a raw (PIL-image) subset.
    Returns metrics dict with tuned per-class thresholds applied.
    """
    model.eval()
    all_labels, all_probs = [], []

    for i in range(len(raw_subset)):
        pil_img, label = raw_subset[i]
        views = [eval_tf(pil_img)]                        # base view
        for _ in range(n_aug - 1):
            views.append(tta_tf(pil_img))                 # augmented views
        batch = torch.stack(views).to(device)             # (n_aug, C, H, W)
        with torch.no_grad():
            avg_probs = torch.sigmoid(model(batch)).mean(dim=0).cpu()
        all_labels.append(label)
        all_probs.append(avg_probs)

    all_labels = torch.stack(all_labels)  # (N, C)
    all_probs  = torch.stack(all_probs)   # (N, C)

    # Apply per-class thresholds
    all_preds_tuned   = (all_probs >= threshold_vec.unsqueeze(0)).float()
    all_preds_default = (all_probs >= 0.5).float()

    m_tuned   = compute_multilabel_metrics(all_labels, all_preds_tuned,   probs=all_probs)
    m_default = compute_multilabel_metrics(all_labels, all_preds_default, probs=all_probs)
    return m_tuned, m_default, all_labels, all_preds_tuned, all_probs

print(f"Running TTA with {TTA_N_AUG} augmented views on test set ({len(test_raw)} images)...")
t0 = time.time()
m_tta_tuned, m_tta_default, test_labels, test_preds_tuned, test_probs = \
    evaluate_tta(best_model, test_raw, TTA_N_AUG, best_thresholds, DEVICE)
print(f"TTA done in {time.time()-t0:.1f}s")

# Also run standard (no TTA) evaluation for comparison
test_loader_eval = DataLoader(TransformSubset(test_raw, eval_tf),
                              batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
best_model.eval()
std_labels, std_preds_all, std_probs_all, std_logits_all = [], [], [], []
with torch.no_grad():
    for imgs, labels in test_loader_eval:
        logits = best_model(imgs.to(DEVICE))
        probs  = torch.sigmoid(logits)
        std_labels.append(labels);       std_probs_all.append(probs.cpu())
        std_logits_all.append(logits.cpu())
std_labels     = torch.cat(std_labels)
std_probs_all  = torch.cat(std_probs_all)
std_logits_all = torch.cat(std_logits_all)
std_preds_05   = (std_probs_all >= 0.5).float()
std_preds_tuned= (std_probs_all >= best_thresholds.unsqueeze(0)).float()

m_std_05    = compute_multilabel_metrics(std_labels, std_preds_05,    probs=std_probs_all, logits=std_logits_all)
m_std_tuned = compute_multilabel_metrics(std_labels, std_preds_tuned, probs=std_probs_all, logits=std_logits_all)

# Print comparison table
rows_cmp = [
    {"config": "baseline (no TTA, t=0.5)",   **{k: round(m_std_05[k],    4) for k in METRIC_KEYS}},
    {"config": "tuned thresholds, no TTA",    **{k: round(m_std_tuned[k], 4) for k in METRIC_KEYS}},
    {"config": "TTA, t=0.5",                  **{k: round(m_tta_default[k],4) for k in METRIC_KEYS}},
    {"config": "TTA + tuned thresholds",      **{k: round(m_tta_tuned[k], 4) for k in METRIC_KEYS}},
]
df_variants = pd.DataFrame(rows_cmp).set_index("config")
print("\n=== Test-set evaluation — variants ===")
print(df_variants[["f1_micro", "exact_match", "hamming_acc", "mean_iou",
                   "precision_micro", "recall_micro", "loss"]].to_string())

## 6 · Detailed Test-Set Analysis

In [ ]:
# Use TTA + tuned thresholds as the final predictions
final_labels = test_labels
final_preds  = test_preds_tuned
final_probs  = test_probs

# Per-class metrics + F1 bar chart
plot_per_class_metrics(final_labels, final_preds)

# Confusion matrices
plot_confusion_matrices(final_labels, final_preds)

# Prediction examples (correct / partial / incorrect)
# Collect normalised images for show_prediction_examples
_imgs_norm = torch.stack([eval_tf(test_raw[i][0]) for i in range(len(test_raw))])
correct_idx, partial_idx, incorrect_idx = categorize_predictions(final_labels, final_preds)
show_prediction_examples(correct_idx,   _imgs_norm, final_labels, final_preds, title="Correct predictions",   n=6)
show_prediction_examples(partial_idx,   _imgs_norm, final_labels, final_preds, title="Partial predictions",   n=6)
show_prediction_examples(incorrect_idx, _imgs_norm, final_labels, final_preds, title="Incorrect predictions", n=4)

## 7 · GradCAM Saliency Maps

We display one representative image per class (same style as notebook 01) with a GradCAM heatmap overlay.  
For ViT (no convolutional layers) we fall back to gradient-based saliency.

In [ ]:
import torch.nn.functional as F

class GradCAM:
    """GradCAM for any differentiable layer (adapted from experiments2/utils.py)."""
    def __init__(self, model, target_layer):
        self.model = model
        self._act  = None
        self._grad = None
        self._fwd  = target_layer.register_forward_hook(lambda m, i, o: self.__setattr__("_act",  o.detach()))
        self._bwd  = target_layer.register_full_backward_hook(lambda m, gi, go: self.__setattr__("_grad", go[0].detach()))

    def __call__(self, img_tensor, device):
        """Return (H, W) heatmap in [0,1]."""
        self.model.eval()
        x = img_tensor.unsqueeze(0).to(device).requires_grad_(True)
        self.model.zero_grad()
        torch.sigmoid(self.model(x)).sum().backward()
        weights = self._grad.mean(dim=[2, 3], keepdim=True)
        cam = F.relu((weights * self._act).sum(dim=1, keepdim=True)).squeeze().cpu()
        cam = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam.numpy()

    def remove(self):
        self._fwd.remove(); self._bwd.remove()


def compute_gradient_saliency(model, img_tensor, device):
    """Gradient saliency fallback (used for ViT)."""
    model.eval()
    x = img_tensor.unsqueeze(0).to(device).requires_grad_(True)
    model.zero_grad()
    torch.sigmoid(model(x)).sum().backward()
    sal = x.grad.abs().squeeze(0).max(dim=0)[0].detach().cpu()
    sal = (sal - sal.min()) / (sal.max() + 1e-8)
    return sal.numpy()


# ── Collect one image per class from the raw train subset (PIL images) ────────
per_class_imgs = {}   # label -> (PIL image, label_tensor)
for pil_img, target in train_raw:
    target_int = target.int()
    for i, label in enumerate(LABEL_ORDER):
        if target_int[i] == 1 and label not in per_class_imgs:
            per_class_imgs[label] = (pil_img.copy(), target.clone())
    if len(per_class_imgs) == NUM_LABELS:
        break

# ── Determine GradCAM or saliency mode ────────────────────────────────────────
gradcam_fn = MODEL_REGISTRY[BEST_ARCH]["gradcam_fn"]
USE_GRADCAM = gradcam_fn is not None
if USE_GRADCAM:
    target_layer = gradcam_fn(best_model)
    cam_fn = GradCAM(best_model, target_layer)
    print(f"GradCAM target layer: {type(target_layer).__name__}")
else:
    print("Using gradient saliency (ViT — no conv layer)")

# ── Plot: one column per class, two rows (original | saliency overlay) ────────
n_cols = len(LABEL_ORDER)
fig, axes = plt.subplots(2, n_cols, figsize=(2.8 * n_cols, 6))

for col, label in enumerate(LABEL_ORDER):
    if label not in per_class_imgs:
        axes[0][col].axis("off"); axes[1][col].axis("off")
        continue

    pil_img, _ = per_class_imgs[label]
    img_t = eval_tf(pil_img)               # normalised tensor (C, H, W)
    img_np = denorm(img_t).permute(1, 2, 0).numpy()
    h, w   = img_t.shape[1], img_t.shape[2]

    # Compute heatmap
    if USE_GRADCAM:
        hmap = cam_fn(img_t, DEVICE)
    else:
        hmap = compute_gradient_saliency(best_model, img_t, DEVICE)

    # Resize heatmap to image dims
    hmap_t = torch.tensor(hmap, dtype=torch.float).unsqueeze(0).unsqueeze(0)
    hmap_up = F.interpolate(hmap_t, size=(h, w), mode="bilinear", align_corners=False)
    hmap_np = hmap_up.squeeze().numpy()

    # Row 0: original
    axes[0][col].imshow(img_np)
    axes[0][col].set_title(label, fontsize=8, pad=3)
    axes[0][col].axis("off")

    # Row 1: heatmap overlay
    axes[1][col].imshow(img_np)
    axes[1][col].imshow(hmap_np, cmap="jet", alpha=0.45)
    axes[1][col].axis("off")

axes[0][0].set_ylabel("Original",    fontsize=9)
axes[1][0].set_ylabel("Saliency" if not USE_GRADCAM else "GradCAM", fontsize=9)
method_name = "Gradient Saliency" if not USE_GRADCAM else "GradCAM"
plt.suptitle(f"{method_name} — one example per class  ({BEST_ARCH})", fontsize=11, y=1.01)
plt.tight_layout(pad=0.3, h_pad=0.5)
plt.show()

if USE_GRADCAM:
    cam_fn.remove()

In [ ]:
# ── Save fine-tuned checkpoint ────────────────────────────────────────────────
OUT_PATH = CKPT_DIR / f"final_finetuned_{BEST_ARCH}.pth"
save_checkpoint(best_model.state_dict(), OUT_PATH)

# ── Final summary ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"  Architecture      : {BEST_ARCH}")
print(f"  Image size        : {IMAGE_SIZE}")
print(f"  Criterion         : AsymmetricLoss (γ⁻=4, γ⁺=1, clip=0.05)")
print(f"  Sampler           : WeightedRandomSampler")
print(f"  Fine-tuning       : Stage A (head) + Stage B (full)" if IS_PRETRAINED else f"  Fine-tuning       : Stage B only (scratch)")
print(f"  TTA views         : {TTA_N_AUG}")
print(f"  Threshold tuning  : per-class (val F1)")
print(f"  Tuned thresholds  : {[round(t, 2) for t in best_thresholds.tolist()]}")
print("-" * 60)
print(f"  {'Metric':<20} {'Before FT':>10} {'After FT (TTA+tuned)':>22}")
print(f"  {'-'*54}")
for k in METRIC_KEYS:
    before = df_cmp.loc[BEST_ARCH, k] if k in df_cmp.columns else float('nan')
    after  = m_tta_tuned[k]
    delta  = after - before
    print(f"  {k:<20} {before:>10.4f} {after:>22.4f}  ({delta:+.4f})")
print("=" * 60)
print(f"\nCheckpoint saved: {OUT_PATH}")